In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import pprint

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

Schemas prevent syntax errors but not semantic ones.  Always use evals and examples to control the quality of the decision

A schema guarantees structure:
- The output has the expected fields
- An enum value comes from the allowed list
- missing_information is output as an array

A schema can't guarantee
- Judgement - e.g may assign normal_return when it should pick damaged_return
- Quality - e.g. may mark urgency: high when nothing justifies it

Exam guidelines:
- Use tool use and JSON schemas when your application needs reliable structured output
- Combine requried fields, nullable, unclear / other, and a detail field
- Use tool_choice to decide wether Claude may use a tool, must use one, or must use a specific one

In [4]:
# Unreliable method - requesting JSON output without providing the structure
message = client.messages.create(
    model=model,
    max_tokens=800,
    output_config={"effort": "low"},
    messages=[
        {
            "role": "user",
            "content": """Extract the return request from this user message.
            Return only valid JSON:
            
            Customer message:
            I got my shoes yesterday and they are scuffed.
            I want a replacement.  I can send a photo if needed.
            """
        }
    ]
)
print(message.content[0].text)

```json
{
  "item": "shoes",
  "issue": "scuffed",
  "request_type": "replacement",
  "delivery_time": "yesterday",
  "photo_available": true
}
```


In [6]:
tools = [
    {
        "name": "extract_return_request",
        "description": "Extract a structured return request from a customer support message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The customer's order ID, or null if not provided.",
                },
                "item": {
                    "type": ["string", "null"],
                    "description": "The item the customer wants to return or replace.",
                },
                "reason": {
                    "type": "string",
                    "enum": ["normal_return", "damaged_item", "billing_dispute",
                             "policy_exception", "unclear", "other"],
                    "description": "The main reason for the request.",
                },
                "reason_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when the reason is other or unclear.",
                },
                "desired_action": {
                    "type": "string",
                    "enum": ["refund", "replacement", "exchange",
                             "store_credit", "unclear", "other"],
                    "description": "What the customer wants as a resolution.",
                },
                "desired_action_detail": {
                    "type": ["string", "null"],
                    "description": "Extra detail when desired_action is unclear or other.",
                },
                "evidence_provided": {
                    "type": "boolean",
                    "description": "Whether the customer already provided evidence, "
                                   "such as photos or proof of purchase.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "normal", "high", "unclear"],
                    "description": "How time-sensitive the request is, based on what "
                                   "the customer states or implies.",
                },
                "missing_information": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Information needed before the request can be processed.",
                },
            },
            "required": [
                "order_id", "item", "reason", "reason_detail",
                "desired_action", "desired_action_detail",
                "evidence_provided", "urgency", "missing_information",
            ],
        },
    }
]

In [7]:
customer_message = """
I got my shoes yesterday and they are scuffed.
I want a replacement.  I can send a photo if needed.
"""

message = client.messages.create(
    model=model,
    max_tokens=800,
    output_config={"effort": "low"},
    tools=tools,
    tool_choice={
        "type": "tool",
        "name": "extract_return_request"
    },
    messages=[
        {
            "role": "user",
            "content": f"""
Extract the return request from this customer message.construct

Use these normalization rules:
- If the customer did not provide an order ID, set order_id to null.
- If the customer says they can provide evidence later, evidence_provided is false.
- Use damaged_item only when the item arrived borken, scratched, scuffed, defective, or unusable.
- Use unclear when the message does not contain enough information to choose confidently.

Customer message:
{customer_message}
"""
        }
    ]
)

tool_use = next(
    block for block in message.content
    if block.type == "tool_use"
)

structured_data = tool_use.input
pprint.pprint(structured_data)


{'desired_action': 'replacement',
 'desired_action_detail': None,
 'evidence_provided': False,
 'item': 'shoes',
 'missing_information': ['order_id', 'photo evidence'],
 'order_id': None,
 'reason': 'damaged_item',
 'reason_detail': 'Shoes arrived scuffed',
 'urgency': 'normal'}


In [8]:
print(message.content[0].model_dump_json(indent=2))

{
  "id": "toolu_01AZSXXrhQg3AjGyNJrWtbWG",
  "caller": {
    "type": "direct"
  },
  "input": {
    "order_id": null,
    "item": "shoes",
    "reason": "damaged_item",
    "reason_detail": "Shoes arrived scuffed",
    "desired_action": "replacement",
    "desired_action_detail": null,
    "evidence_provided": false,
    "urgency": "normal",
    "missing_information": [
      "order_id",
      "photo evidence"
    ]
  },
  "name": "extract_return_request",
  "type": "tool_use"
}
